# RepLens — 02: Dataset preparation

Validate reviewed annotations, define past-only video windows and prepare participant-grouped evaluation folds. Inputs are original video frames; MediaPipe measurements are excluded. RIR labels retain their endpoint provenance. This is a three-person pilot, not an accuracy result.

Prepare all inventoried sources. Confirmed labels form the training set; automatic candidates without targets form a separate prediction-only set.

`VIDEO_VARIANT` selects original or whole-lifter crop copies. Original labels are validated first; verified crop fingerprints are then attached to the prepared windows. Crops were planned from sampled frames across each complete clip: this is retrospective preprocessing, not a validated live crop procedure.

## 1. Setup

Run cells in order. The reviewed table must exist from data review. Generated indexes and reports stay under ignored `training/data/prepared/`; original videos and annotations are unchanged.

In [ ]:
from pathlib import Path
import hashlib
import json
import math

import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents]
             if (path / ".git").is_dir() and (path / "training" / "notebooks").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open the notebook from the RepLens project folder.")
DATA_DIR = ROOT / "training" / "data"
VIDEO_VARIANT = "cropped"
if VIDEO_VARIANT not in {"original", "cropped"}:
    raise ValueError("Choose original or cropped video framing.")
SOURCE_VIDEO_DIR = DATA_DIR / "videos"
VIDEO_DIR = SOURCE_VIDEO_DIR
CROP_EXPORT_PATH = DATA_DIR / "review/whole_lifter_crops/exports.json"
LABEL_PATH = DATA_DIR / "processed" / "reviewed_reps.csv"
CANDIDATE_PATH = DATA_DIR / "processed" / "candidate_reps.csv"
BATCH_PATH = DATA_DIR / "review" / "batch_summary.csv"
MANIFEST_PATH = DATA_DIR / "video_manifest.csv"
DATASET_SCOPE = "exploratory"
if DATASET_SCOPE not in {"exploratory", "confirmed"}:
    raise ValueError("Choose exploratory or confirmed dataset scope.")
HISTORY_REPS = 4
OUTPUT_DIR = DATA_DIR / "prepared" / f"history_{HISTORY_REPS}_reps"
if not LABEL_PATH.is_file() or not MANIFEST_PATH.is_file():
    raise FileNotFoundError("Run data review and save the reviewed rep table first.")
if type(HISTORY_REPS) is not int or HISTORY_REPS < 1:
    raise ValueError("HISTORY_REPS must be a positive integer.")

### Validation helpers

Checks cover source hashes, labels, timestamps and fold separation. Video timestamps are read from the decoder; reported FPS alone is not used to locate windows.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_reviewed_dataset(label_path, manifest_path):
    labels = pd.read_csv(label_path, dtype={"video_id": str, "participant_id": str, "set_id": str})
    inventory = pd.read_csv(manifest_path, dtype=str).fillna("")
    required = {"video_id", "participant_id", "set_id", "rep", "start_s", "top_s", "end_s", "relative_path", "video_sha256", "rir_label_status", "reps_left", "endpoint_source", "endpoint_evidence"}
    missing = required - set(labels.columns)
    if missing:
        raise ValueError(f"Missing annotation columns: {sorted(missing)}")
    if DATASET_SCOPE == "confirmed":
        labels = labels[labels.rir_label_status.isin(["collector_confirmed_failure", "reviewer_confirmed_failure"])].copy()
    if labels.empty:
        raise ValueError("The reviewed table contains no repetitions.")
    for column in ["video_id", "participant_id", "set_id", "relative_path", "video_sha256", "endpoint_source", "endpoint_evidence"]:
        if labels[column].isna().any() or labels[column].astype(str).str.strip().eq("").any():
            raise ValueError(f"Missing {column} in reviewed annotations.")
    for column in ["rep", "start_s", "top_s", "end_s", "reps_left"]:
        labels[column] = pd.to_numeric(labels[column], errors="raise")
        if not np.isfinite(labels[column].to_numpy(dtype=float)).all() or labels[column].lt(0).any():
            raise ValueError(f"Invalid numeric values in {column}.")
    for column in ["rep", "reps_left"]:
        if not np.equal(labels[column], np.floor(labels[column])).all():
            raise ValueError(f"{column} must contain integers.")
        labels[column] = labels[column].astype(int)
    accepted = ["collector_confirmed_failure", "reviewer_confirmed_failure"]
    if DATASET_SCOPE == "exploratory":
        accepted.append("user_reported_failure")
    if not labels.rir_label_status.isin(accepted).all():
        raise ValueError("Exclude or resolve unconfirmed labels before preparation.")
    if labels.duplicated(["participant_id", "set_id", "rep"]).any():
        raise ValueError("Duplicate participant/set/rep annotations.")
    if inventory.video_id.duplicated().any() or inventory.relative_path.duplicated().any():
        raise ValueError("Duplicate entries in the video inventory.")
    for video_id, group in labels.groupby("video_id", sort=False):
        matches = inventory[inventory.video_id == video_id]
        if len(matches) != 1:
            raise ValueError(f"Expected one inventory entry for {video_id}.")
        record = matches.iloc[0]
        if record.reuse_status != "confirmed" or not record.reuse_evidence.strip():
            exploratory_source = (DATASET_SCOPE == "exploratory" and group.rir_label_status.eq("user_reported_failure").all()
                                  and record.source_url.startswith("https://www.youtube.com/")
                                  and record.reuse_status == "pending" and bool(record.reuse_evidence.strip()))
            if not exploratory_source:
                raise ValueError(f"Reuse/consent evidence is missing for {video_id}.")
        for column in ["participant_id", "relative_path"]:
            if group[column].nunique() != 1 or group[column].iloc[0] != record[column]:
                raise ValueError(f"Inventory and annotations disagree on {column} for {video_id}.")
        if group.set_id.nunique() != 1 or group.video_sha256.nunique() != 1:
            raise ValueError("A video must identify one set and one source hash in this pilot.")
        source = (VIDEO_DIR / record.relative_path).resolve()
        if not source.is_relative_to(VIDEO_DIR.resolve()) or not source.is_file():
            raise ValueError(f"Missing or invalid source path for {video_id}.")
        if sha256_file(source) != group.video_sha256.iloc[0]:
            raise ValueError(f"Source changed after review: {video_id}.")
        ordered = group.sort_values("rep")
        if ordered.rep.tolist() != list(range(1, len(ordered) + 1)):
            raise ValueError(f"Non-contiguous rep numbers for {video_id}.")
        if ordered.reps_left.tolist() != list(range(len(ordered) - 1, -1, -1)):
            raise ValueError(f"RIR labels do not match the reviewed endpoint for {video_id}.")
        times = ordered[["start_s", "top_s", "end_s"]].to_numpy()
        if not ((times[:, 0] < times[:, 1]) & (times[:, 1] < times[:, 2])).all():
            raise ValueError(f"Invalid rep boundaries for {video_id}.")
        if len(times) > 1 and (times[1:, 0] < times[:-1, 2] - 0.02).any():
            raise ValueError(f"Overlapping or unordered reps for {video_id}.")
    if labels.groupby("video_sha256").participant_id.nunique().gt(1).any():
        raise ValueError("Identical video bytes are assigned to different people.")
    return labels.sort_values(["participant_id", "set_id", "rep"]).reset_index(drop=True), inventory


def scan_video_timestamps(path):
    capture = cv2.VideoCapture(str(path))
    times = []
    try:
        if not capture.isOpened():
            raise ValueError("Video cannot be opened.")
        width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
        reported_fps = float(capture.get(cv2.CAP_PROP_FPS))
        while capture.grab():
            times.append(float(capture.get(cv2.CAP_PROP_POS_MSEC)) / 1000)
    finally:
        capture.release()
    values = np.asarray(times, dtype=np.float64)
    if len(values) < 2 or not np.isfinite(values).all() or values[0] < 0 or not (np.diff(values) > 0).all():
        raise ValueError("Decoder timestamps are missing or non-monotonic; timing needs review.")
    delta = np.diff(values)
    median_delta = float(np.median(delta))
    info = {"decoded_frames": len(values), "width": width, "height": height,
            "reported_fps": reported_fps, "median_decoded_fps": 1 / median_delta,
            "last_frame_s": float(values[-1]), "largest_frame_gap_s": float(delta.max()),
            "gaps_over_1_5x_median": int((delta > 1.5 * median_delta).sum())}
    return values, info


def make_windows(labels, timestamps, history_reps):
    rows = []
    for video_id, group in labels.groupby("video_id", sort=False):
        ordered = group.sort_values("rep").reset_index(drop=True)
        times = timestamps[video_id]
        if ordered.end_s.max() > times[-1] + 0.02:
            raise ValueError(f"Annotations extend beyond decoded frames for {video_id}.")
        for position, rep in ordered.iterrows():
            first = max(0, position - history_reps + 1)
            start = float(ordered.iloc[first].start_s)
            end = float(rep.end_s)
            start_frame = int(np.searchsorted(times, start, side="left"))
            stop_frame = int(np.searchsorted(times, end, side="right"))
            if stop_frame <= start_frame:
                raise ValueError(f"Empty video window for {video_id}, rep {rep.rep}.")
            identity = f"{rep.video_sha256}|{rep.rep}|{start:.6f}|{end:.6f}|{history_reps}"
            rows.append({"sample_id": hashlib.sha256(identity.encode()).hexdigest()[:24],
                         "video_id": video_id, "participant_id": rep.participant_id, "set_id": rep.set_id,
                         "rep": int(rep.rep), "relative_path": rep.relative_path, "video_sha256": rep.video_sha256,
                         "clip_start_s": start, "clip_end_s": end,
                         "frame_start": start_frame, "frame_stop_exclusive": stop_frame,
                         "available_history_reps": position - first + 1,
                         "reps_left": int(rep.reps_left) if pd.notna(rep.reps_left) else np.nan, "rir_label_status": rep.rir_label_status,
                         "endpoint_source": rep.endpoint_source, "endpoint_evidence": rep.endpoint_evidence})
    result = pd.DataFrame(rows)
    if result.sample_id.duplicated().any():
        raise ValueError("Duplicate training windows.")
    return result


def participant_folds(windows):
    people = sorted(windows.participant_id.unique().tolist())
    if len(people) < 3:
        raise ValueError("This nested pilot plan requires at least three distinct participants.")
    folds = []
    for test_person in people:
        outer_train = [person for person in people if person != test_person]
        inner = []
        for validation_person in outer_train:
            train_people = [person for person in outer_train if person != validation_person]
            inner.append({"train_participants": train_people, "validation_participants": [validation_person]})
        folds.append({"fold_id": f"test_{test_person}", "outer_train_participants": outer_train,
                      "test_participants": [test_person], "inner_validation": inner})
    return folds

def read_candidate_dataset(path, inventory):
    if not path.exists():
        raise FileNotFoundError("Run the updated batch cell in notebook 01 to export candidate_reps.csv.")
    candidates = pd.read_csv(path, dtype={"video_id": str, "participant_id": str, "set_id": str}).fillna({"participant_id": ""})
    required = {"video_id", "participant_id", "set_id", "relative_path", "video_sha256", "rep", "start_s", "top_s", "end_s", "reps_left", "rir_label_status", "endpoint_source", "endpoint_evidence"}
    if not required.issubset(candidates.columns):
        raise ValueError("Candidate table is missing required columns. Rerun notebook 01.")
    if candidates.reps_left.notna().any() or not candidates.rir_label_status.eq("unlabeled").all():
        raise ValueError("Automatic candidates must not invent RIR targets.")
    for video_id, group in candidates.groupby("video_id", sort=False):
        match = inventory[inventory.video_id == video_id]
        if len(match) != 1:
            raise ValueError(f"Candidate inventory mismatch: {video_id}")
        record = match.iloc[0]
        if group.relative_path.nunique() != 1 or group.relative_path.iloc[0] != record.relative_path:
            raise ValueError(f"Candidate source path mismatch: {video_id}")
        if group.participant_id.nunique() != 1 or group.participant_id.iloc[0] != record.participant_id:
            raise ValueError(f"Candidate participant mismatch: {video_id}. Rerun notebook 01.")
        if group.video_sha256.nunique() != 1 or sha256_file(VIDEO_DIR / record.relative_path) != group.video_sha256.iloc[0]:
            raise ValueError(f"Stale candidate video: {video_id}")
        values = group[["start_s", "top_s", "end_s"]].to_numpy(dtype=float)
        if not np.isfinite(values).all() or not ((0 <= values[:, 0]) & (values[:, 0] < values[:, 1]) & (values[:, 1] < values[:, 2])).all():
            raise ValueError(f"Invalid candidate boundaries: {video_id}")
    return candidates




## 2. Validate the reviewed data

Counts and labels are checked against the recorded endpoint definition. This verifies consistency, not physiological truth. Consent and endpoints remain collector-confirmed where indicated.

In [ ]:
inventory_status = pd.read_csv(MANIFEST_PATH, dtype=str).fillna("")
saved_labels = pd.read_csv(LABEL_PATH)
reviewed_counts = saved_labels.groupby("video_id").size()
coverage = inventory_status[["video_id", "relative_path", "participant_id", "reuse_status", "label_status"]].copy()
coverage["saved_reviewed_reps"] = coverage.video_id.map(reviewed_counts).fillna(0).astype(int)
coverage["preparation_status"] = np.where(coverage.saved_reviewed_reps.gt(0), "Saved labels available", "Pending reviewed labels")
coverage.loc[coverage.label_status.eq("unlabeled_nonfailure"), "preparation_status"] = "Nonfailure window — no RIR target"
print(f"Inventory: {len(coverage)} videos | Saved labels: {(coverage.saved_reviewed_reps > 0).sum()} videos")
display(coverage)
if coverage.saved_reviewed_reps.eq(0).any():
    print("Unlabeled clips are prepared for experimental prediction only. Supported endpoint labels and documented reuse evidence are needed before adding them to the supervised benchmark. Keep separate sets separate.")

labels, inventory = read_reviewed_dataset(LABEL_PATH, MANIFEST_PATH)
candidate_labels = read_candidate_dataset(CANDIDATE_PATH, inventory)
if VIDEO_VARIANT == "cropped":
    export_records = json.loads(CROP_EXPORT_PATH.read_text())
    crop_exports = {row["filename"]: row for row in export_records}
    cropped_dir = DATA_DIR / "cropped_videos"
    for table in [labels, candidate_labels]:
        for filename, group in table.groupby("relative_path"):
            record = crop_exports.get(filename)
            if record is None:
                raise ValueError(f"No verified crop export: {filename}")
            if set(group.video_sha256) != {record["source_sha256"]}:
                raise ValueError(f"Crop was made from a different source: {filename}")
            if sha256_file(cropped_dir / filename) != record["output_sha256"]:
                raise ValueError(f"Cropped video changed after verification: {filename}")
            table.loc[group.index, "video_sha256"] = record["output_sha256"]
    VIDEO_DIR = cropped_dir
print("Video framing:", VIDEO_VARIANT)
inference_labels = candidate_labels.loc[~candidate_labels.video_id.isin(labels.video_id)].copy()
print(f"Additional automatic reps for experimental prediction only: {len(inference_labels)} across {inference_labels.video_id.nunique()} clips. No RIR accuracy is computed for these rows.")
print("Dataset scope:", DATASET_SCOPE)
if DATASET_SCOPE == "exploratory":
    print("Includes user-reported endpoints and public sources with unverified reuse permission. Results are exploratory, not an approved benchmark.")
print(f"Reviewed reps: {len(labels)} | Videos: {labels.video_id.nunique()} | People: {labels.participant_id.nunique()}")
display(labels.groupby(["participant_id", "set_id"]).agg(
    completed_reps=("rep", "size"), min_rir=("reps_left", "min"), max_rir=("reps_left", "max")
))
display(labels.groupby(["rir_label_status", "endpoint_source"]).size().rename("repetitions"))

## 3. Audit frame timing

Decode timestamps once per video without loading frames into memory. Missing/non-monotonic timing blocks preparation. Large frame gaps are flagged for investigation. This does not verify camera exposure, rolling shutter or physiological tremor visibility.

In [ ]:
timestamps = {}
video_info = []
required_video_ids = set(labels.video_id) | set(inference_labels.video_id)
active_inventory = inventory.loc[inventory.video_id.isin(required_video_ids)]
assert set(active_inventory.video_id) == required_video_ids
print(f"Scanning {len(active_inventory)} videos used by training/prediction windows; other inventory entries remain pending.")
for record in active_inventory.itertuples(index=False):
    print("Reading timestamps:", record.relative_path, flush=True)
    source = (VIDEO_DIR / record.relative_path).resolve()
    if not source.is_relative_to(VIDEO_DIR.resolve()) or not source.is_file():
        raise ValueError(f"Missing source: {record.relative_path}")
    times, info = scan_video_timestamps(source)
    timestamps[record.video_id] = times
    video_info.append({"video_id": record.video_id, "relative_path": record.relative_path, **info})
video_info = pd.DataFrame(video_info)
display(video_info)
if video_info.gaps_over_1_5x_median.gt(0).any():
    print("Timing warning: inspect flagged videos before motion-frequency analysis.")


## 4. Define video windows

Each example ends at a reviewed completed rep and includes that rep plus up to three earlier reps. Short histories stay short; no future frames or final set length enter the video input. Windows overlap, so they are not independent participants.

No resized videos or low-FPS training clips are written. The index preserves native frame references; model preprocessing and any temporal downsampling must be justified in training, particularly for small-motion analysis.

In [ ]:
windows = make_windows(labels, timestamps, HISTORY_REPS)
display(windows[["sample_id", "participant_id", "set_id", "rep", "clip_start_s", "clip_end_s", "available_history_reps", "reps_left"]].head(10))
print(f"Prepared window references: {len(windows)}")
print("Model input: original video frames. IDs and annotation metadata are not image-model features.")
inference_windows = make_windows(inference_labels, timestamps, HISTORY_REPS) if not inference_labels.empty else windows.iloc[:0].copy()
if not inference_windows.empty:
    assert inference_windows.reps_left.isna().all()
    assert set(inference_windows.video_id).isdisjoint(set(windows.video_id))
print(f"Prediction-only windows: {len(inference_windows)}. These do not enter training or accuracy scores.")


### Optional window preview

Set `PREVIEW_SAMPLE` to a 1-based row number. Eight frames are sampled for display only; this is not the model's training sampling policy. Clear image outputs before sharing.

In [ ]:
PREVIEW_SAMPLE = None

if PREVIEW_SAMPLE is None:
    print("Set PREVIEW_SAMPLE to inspect one window.")
elif type(PREVIEW_SAMPLE) is not int or not 1 <= PREVIEW_SAMPLE <= len(windows):
    raise ValueError("Choose a valid 1-based window row.")
else:
    window = windows.iloc[PREVIEW_SAMPLE - 1]
    indices = np.unique(np.linspace(window.frame_start, window.frame_stop_exclusive - 1, 8).astype(int))
    capture = cv2.VideoCapture(str(VIDEO_DIR / window.relative_path))
    figure, axes = plt.subplots(1, len(indices), figsize=(3 * len(indices), 4), squeeze=False)
    try:
        for axis, index in zip(axes[0], indices):
            capture.set(cv2.CAP_PROP_POS_FRAMES, int(index))
            ok, frame = capture.read()
            axis.axis("off")
            axis.set_title(f"{timestamps[window.video_id][index]:.2f}s")
            if not ok:
                raise ValueError("Could not read a preview frame.")
            axis.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        plt.tight_layout()
        plt.show()
    finally:
        capture.release()
        plt.close(figure)

## 5. Participant-separated evaluation plan

Each person is held out once for testing. Only the other people are used for training and validation. With three people, each inner split has one training person and one validation person; later final fitting can use both outer-training people. Choose settings without consulting test results.

Three people support a small feasibility experiment. Many clips from those people do not establish broad generalization.

In [ ]:
folds = participant_folds(windows)
fold_rows = []
for fold in folds:
    test = set(fold["test_participants"])
    outer = set(fold["outer_train_participants"])
    assert test.isdisjoint(outer)
    for inner_id, inner in enumerate(fold["inner_validation"], start=1):
        train = set(inner["train_participants"])
        validation = set(inner["validation_participants"])
        assert train.isdisjoint(validation) and train.isdisjoint(test) and validation.isdisjoint(test)
        subsets = [windows[windows.participant_id.isin(group)] for group in [train, validation, test]]
        for a, b in [(0, 1), (0, 2), (1, 2)]:
            assert set(subsets[a].video_sha256).isdisjoint(subsets[b].video_sha256)
        fold_rows.append({"outer_fold": fold["fold_id"], "inner_split": inner_id,
                          "train_people": ", ".join(sorted(train)), "validation_people": ", ".join(sorted(validation)),
                          "test_people": ", ".join(sorted(test)),
                          "train_windows": len(subsets[0]), "validation_windows": len(subsets[1]), "test_windows": len(subsets[2])})
display(pd.DataFrame(fold_rows))

## 6. Save the preparation outputs

The preparation ID fingerprints annotations, inventory, configuration and window/fold definitions. Rerunning the same inputs reuses the same location; changed inputs get a separate directory. Frame timestamp arrays are saved alongside the video index. Original videos and reviewed labels remain unchanged.

In [ ]:
configuration = {"history_reps": HISTORY_REPS, "dataset_scope": DATASET_SCOPE, "label_sha256": sha256_file(LABEL_PATH),
                 "inventory_sha256": sha256_file(MANIFEST_PATH), "candidate_sha256": sha256_file(CANDIDATE_PATH), "format_version": "video-windows-v3", "video_variant": VIDEO_VARIANT,
                 "video_directory": VIDEO_DIR.name,
                 "crop_exports_sha256": sha256_file(CROP_EXPORT_PATH) if VIDEO_VARIANT == "cropped" else None}
window_bytes = windows.to_csv(index=False).encode()
fold_bytes = json.dumps(folds, sort_keys=True).encode()
inference_bytes = inference_windows.to_csv(index=False).encode()
identity = json.dumps(configuration, sort_keys=True).encode() + window_bytes + inference_bytes + fold_bytes
preparation_id = hashlib.sha256(identity).hexdigest()[:16]
run_dir = OUTPUT_DIR / preparation_id
run_dir.mkdir(parents=True, exist_ok=True)
(run_dir / "windows.csv").write_bytes(window_bytes)
(run_dir / "inference_windows.csv").write_bytes(inference_bytes)
video_info.to_csv(run_dir / "videos.csv", index=False)
(run_dir / "folds.json").write_text(json.dumps(folds, indent=2) + "\n")
np.savez_compressed(run_dir / "timestamps.npz", **timestamps)
report = {"preparation_id": preparation_id, "configuration": configuration,
          "windows": len(windows), "videos": labels.video_id.nunique(), "participants": labels.participant_id.nunique(),
          "endpoint_sources": labels.endpoint_source.value_counts().to_dict(),
          "timing_warnings": video_info.loc[video_info.gaps_over_1_5x_median.gt(0), "video_id"].tolist(),
          "dataset_scope": DATASET_SCOPE, "user_reported_reps": int(labels.rir_label_status.eq("user_reported_failure").sum()),
          "unverified_reuse_video_ids": inventory.loc[inventory.video_id.isin(labels.video_id) & inventory.reuse_status.ne("confirmed"), "video_id"].tolist(),
          "inventory_videos": len(inventory), "prediction_only_windows": len(inference_windows),
          "prediction_only_videos": int(inference_windows.video_id.nunique()),
          "videos_without_windows": sorted(set(inventory.video_id) - set(windows.video_id) - set(inference_windows.video_id)),
          "model_input": "original RGB video frames; no pose features", "trained_model": False}
(run_dir / "preparation.json").write_text(json.dumps(report, indent=2) + "\n")
print("Saved:", run_dir.relative_to(ROOT))
print("Next: train the video baseline and log experiments to W&B using these fixed folds.")
coverage["candidate_reps"] = coverage.video_id.map(candidate_labels.groupby("video_id").size()).fillna(0).astype(int)
coverage["prediction_windows"] = coverage.video_id.map(inference_windows.groupby("video_id").size()).fillna(0).astype(int)
coverage.loc[coverage.saved_reviewed_reps.eq(0) & coverage.prediction_windows.gt(0), "preparation_status"] = "Prediction only — no verified target"
coverage.loc[coverage.saved_reviewed_reps.eq(0) & coverage.prediction_windows.eq(0), "preparation_status"] = "No candidate reps — needs tracking/boundary review"
coverage.loc[coverage.video_id.isin(labels.loc[labels.rir_label_status.eq("user_reported_failure"), "video_id"]), "preparation_status"] = "Exploratory training — user-reported failure; reuse unverified"
coverage.to_csv(run_dir / "coverage.csv", index=False)
display(coverage)

if BATCH_PATH.exists():
    batch_status = pd.read_csv(BATCH_PATH).fillna("")
    coverage["processing_status"] = coverage.relative_path.map(batch_status.set_index("relative_path").status).fillna("not_processed")
    failed = ~coverage.processing_status.isin(["processed", "cached"])
    coverage.loc[failed, "preparation_status"] = "Processing incomplete — inspect notebook 01"
    coverage.to_csv(run_dir / "coverage.csv", index=False)
